In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
import pandas as pd
import duckdb

# Import a downloaded .csv from Kaggle that records the race results from Australia to Spain in 2025 season

df = pd.read_csv("/kaggle/input/datasets/lawrencenieto/f1-2025-race-results/F1_2025_RaceResults.csv")

# Create a raw_race_results table from the df via copy
raw_race_results = df.copy()

raw_race_results.info()
raw_race_results.head()

# create an in-memory SQL database and register the table
con = duckdb.connect(database=':memory:') # `duckdb.connect(database=':memory:')` → “make a tiny fake database in RAM.”
con.register('raw_race_results', df) # treat the CSV as a SQL table named raw_rcce_results


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 179 entries, 0 to 178
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Track             179 non-null    object
 1   Position          179 non-null    object
 2   No                179 non-null    int64 
 3   Driver            179 non-null    object
 4   Team              179 non-null    object
 5   Starting Grid     179 non-null    int64 
 6   Laps              179 non-null    int64 
 7   Time/Retired      179 non-null    object
 8   Points            179 non-null    int64 
 9   Set Fastest Lap   179 non-null    object
 10  Fastest Lap Time  173 non-null    object
dtypes: int64(4), object(7)
memory usage: 15.5+ KB


In [4]:
con.execute("""
    SELECT
        "Track",
        "Driver",
        "Team",
        "Position",
        "Points"
    FROM raw_race_results
    LIMIT 5
""").df()


,Track,Driver,Team,Position,Points
0,Australia,Lando Norris,McLaren Mercedes,1,25
1,Australia,Max Verstappen,Red Bull Racing Honda RBPT,2,18
2,Australia,George Russell,Mercedes,3,15
3,Australia,Kimi Antonelli,Mercedes,4,12
4,Australia,Alexander Albon,Williams Mercedes,5,10


In [5]:
# Create a clean fact table from raw_race_results
con.execute("""
    CREATE TABLE fact_driver_race_results AS
    SELECT
        "Track"             AS track,
        "Driver"            AS driver_name,
        "Team"              AS team_name,
        "Position"          AS position,
        "No"                AS car_no,
        "Starting Grid"     AS starting_grid,
        "Laps"              AS laps,
        "Time/Retired"      AS time_or_retired,
        "Points"            AS points,
        "Set Fastest Lap"   AS set_fastest_lap,
        "Fastest Lap Time"  AS fastest_lap_time
    FROM raw_race_results
""")

con.execute("""
    SELECt *
    FROM fact_driver_race_results
    LIMIT 5
    """).df()

,track,driver_name,team_name,position,car_no,starting_grid,laps,time_or_retired,points,set_fastest_lap,fastest_lap_time
0,Australia,Lando Norris,McLaren Mercedes,1,4,1,57,1:42:06.304,25,Yes,1:22.167
1,Australia,Max Verstappen,Red Bull Racing Honda RBPT,2,1,3,57,+0.895,18,No,1:23.081
2,Australia,George Russell,Mercedes,3,63,4,57,+8.481,15,No,1:25.065
3,Australia,Kimi Antonelli,Mercedes,4,12,16,57,+10.135,12,No,1:24.901
4,Australia,Alexander Albon,Williams Mercedes,5,23,6,57,+12.773,10,No,1:24.597


In [6]:
# Create a dimension table from our fact table dim_drivers

con.execute("""
    CREATE TABLE dim_driver AS
    SELECT
        ROW_NUMBER() OVER (ORDER BY driver_name) AS driver_id,
        driver_name
    FROM fact_driver_race_results
    GROUP BY driver_name
""")

con.execute("""
    SELECT * 
    FROM dim_driver
    LIMIT 5
""").df()

,driver_id,driver_name
0,1,Alexander Albon
1,2,Carlos Sainz
2,3,Charles Leclerc
3,4,Esteban Ocon
4,5,Fernando Alonso


In [7]:
# Build a final fact table with dim_table and fact_race_results

con.execute("""
    CREATE TABLE fact_driver_race_results_with_ids AS
    SELECT
        f.track,
        f.team_name,
        f.position,
        f.car_no,
        f.starting_grid,
        f.laps,
        f.time_or_retired,
        f.points,
        f.set_fastest_lap,
        f.fastest_lap_time,
        d.driver_id,
        f.driver_name
    FROM fact_driver_race_results f
    JOIN dim_driver d
      ON f.driver_name = d.driver_name
""")

con.execute("""
    SELECT *
    FROM fact_driver_race_results_with_ids
    LIMIT 5
""").df()

,track,team_name,position,car_no,starting_grid,laps,time_or_retired,points,set_fastest_lap,fastest_lap_time,driver_id,driver_name
0,Australia,McLaren Mercedes,1,4,1,57,1:42:06.304,25,Yes,1:22.167,13,Lando Norris
1,Australia,Red Bull Racing Honda RBPT,2,1,3,57,+0.895,18,No,1:23.081,16,Max Verstappen
2,Australia,Mercedes,3,63,4,57,+8.481,15,No,1:25.065,8,George Russell
3,Australia,Mercedes,4,12,16,57,+10.135,12,No,1:24.901,11,Kimi Antonelli
4,Australia,Williams Mercedes,5,23,6,57,+12.773,10,No,1:24.597,1,Alexander Albon


In [8]:
# Ensure the transofmration didn't lose any rows by comparing old table with new table

con.execute("""
    SELECT
        (SELECT COUNT(*) FROM fact_driver_race_results) AS fact_base_count,
        (SELECT COUNT(*) FROM fact_driver_race_results_with_ids) AS fact_with_ids_count
""").df()


,fact_base_count,fact_with_ids_count
0,179,179


In [9]:
# Check for empty(null) driver_id's
con.execute("""
    SELECT COUNT(*) AS no_of_null_driver_ids
    FROM fact_driver_race_results_with_ids
    WHERE driver_id IS NULL
""").df()


,no_of_null_driver_ids
0,0


In [10]:
# Let's find total points per driver so we can determine who won the F1 2025 driver championship

con.execute("""
    SELECT SUM(points) AS total_points, driver_name, driver_id
    FROM fact_driver_race_results_with_ids
    GROUP BY driver_id, driver_name
    ORDER BY SUM(points) DESC
    """).df()

# normally, you would use a join to join dim_drivers and fact, but because we already have a fact table with ids, we can just do this.
# if we decided to add more data (columns) to dim_drivers, that's why we would keep the dim table.

,total_points,driver_name,driver_id
0,172.0,Oscar Piastri,19
1,167.0,Lando Norris,13
2,131.0,Max Verstappen,16
3,101.0,George Russell,8
4,90.0,Charles Leclerc,3
5,57.0,Lewis Hamilton,14
6,44.0,Kimi Antonelli,11
7,42.0,Alexander Albon,1
8,21.0,Isack Hadjar,9
9,20.0,Esteban Ocon,4
